# Health v7
This adds stop tokens to the training

# Load model with Unsloth patching

In [1]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    "peers-ai/wtk-qwen3-8b-abliterated-health-merged-v6",
    max_seq_length=1024,
    load_in_4bit=True,
)

print("Loaded model in 4-bit ✅")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
INFO 11-20 16:48:55 [__init__.py:216] Automatically detected platform cuda.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.11.2: Fast Qwen3 patching. Transformers: 4.57.0. vLLM: 0.10.2.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.318 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33+e98c69b.d20251109. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.90G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loaded model in 4-bit ✅


# Apply LoRa adapter

In [4]:
peft_model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Keep: balances capacity and efficiency
    lora_alpha=32,  # Lowered: reduces overfitting risk (scale = 1)
    lora_dropout=0.2,  # Slightly higher: better regularization for noisy esoteric texts
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"  # Attention: core for reasoning
        #"up_proj", "down_proj", "gate_proj"  # MLP: adds expressivity for complex patterns
    ],
    bias="none",  # Keep: minimizes params
    use_gradient_checkpointing=True,  # Keep: VRAM saver
    modules_to_save=None,  # Default: avoids retraining embeddings
    use_rslora=False  # NEW: rank-stabilized LoRA, improves stability for higher r
)

# --- AFTER loading model & tokenizer ---
# Let the tokenizer keep its native tokens (Llama-3.2)
# DO NOT override BOS — it should be "<s>" (ID 0)
# EOS is already "</s>" (ID 1)

# Only set PAD if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Safe: use "</s>" as pad

# Sync config (critical for Trainer)
model.config.bos_token_id = tokenizer.bos_token_id      # 0
model.config.eos_token_id = tokenizer.eos_token_id      # 1
model.config.pad_token_id = tokenizer.pad_token_id      # 1

peft_model.config.bos_token_id = tokenizer.bos_token_id
peft_model.config.eos_token_id = tokenizer.eos_token_id
peft_model.config.pad_token_id = tokenizer.pad_token_id

print("Loaded peft model ✅")


Unsloth 2025.11.2 patched 36 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Loaded peft model ✅


# Load the dataset from corpus

In [5]:
import os 
import json

CORPUS_QA_DIR = "/storage/corpus/corpus_health_tiny_qa"
BLOCK_SIZE = 1024  # max tokens per chunk
EOS_STR = "</s>"

tok = tokenizer 

# Ensure EOS/PAD exist and are consistent
_added = False
if tok.eos_token is None:
    tok.add_special_tokens({"eos_token": EOS_STR})
    _added = True
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
    _added = True
if _added:
    model.resize_token_embeddings(len(tok))

# -----------------------
# 1) Load QA JSON files (array JSON or JSONL), normalize to strings
# -----------------------
def _normalize_one(q: str, a: str) -> str:
    q = (q or "").strip()
    a = (a or "").strip()
    # If answer already has </s>, remove so we enforce exactly one.
    if a.endswith(EOS_STR):
        a = a[: -len(EOS_STR)].rstrip()
    return f"Q: {q}\nA: {a}{EOS_STR}"

def load_qa_corpus(directory):
    texts = []
    for filename in os.listdir(directory):
        if not (filename.endswith(".json") or filename.endswith(".jsonl")):
            continue
        path = os.path.join(directory, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            head = f.read(1)
            f.seek(0)
            if head == "[":  # JSON array
                try:
                    arr = json.load(f)
                    for obj in arr:
                        if isinstance(obj, dict) and "question" in obj and "answer" in obj:
                            texts.append(_normalize_one(obj["question"], obj["answer"]))
                except json.JSONDecodeError:
                    # fall back to line-by-line if malformed
                    f.seek(0)
                    for line in f:
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            obj = json.loads(line)
                            if "question" in obj and "answer" in obj:
                                texts.append(_normalize_one(obj["question"], obj["answer"]))
                        except json.JSONDecodeError:
                            continue
            else:  # JSONL
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        obj = json.loads(line)
                        if "question" in obj and "answer" in obj:
                            texts.append(_normalize_one(obj["question"], obj["answer"]))
                    except json.JSONDecodeError:
                        continue
    return texts

raw_texts = load_qa_corpus(CORPUS_QA_DIR)
print(f"Loaded {len(raw_texts)} QA items from {CORPUS_QA_DIR} ✅")

Loaded 30626 QA items from /storage/corpus/corpus_health_tiny_qa ✅


# Tokenize with EOS appended (token id, not string)
We’ll build one long stream of ids and then pack into BLOCK_SIZE chunks.

In [7]:
from datasets import Dataset

# -------------------------------------------------
# 1. Tokenize + append EOS (one token per example)
# -------------------------------------------------
def tokenize_append_eos(texts):
    """
    texts: str   OR   list[str]
    Returns: flat list[int] of token ids with an EOS after every example.
    """
    # ------------------------------------------------------------------
    # 1. Make sure we always have a list of strings
    # ------------------------------------------------------------------
    if isinstance(texts, str):
        texts = [texts]                     # single example → batch of 1
    elif not isinstance(texts, list):
        raise TypeError("`texts` must be str or list[str]")

    # ------------------------------------------------------------------
    # 2. Batch-tokenize (no BOS/EOS – we add EOS ourselves)
    # ------------------------------------------------------------------
    enc = tok(
        texts,
        add_special_tokens=False,          # we control EOS ourselves
        padding=False,
        truncation=False,
        return_attention_mask=False,
    )

    # ------------------------------------------------------------------
    # 3. Flatten + append EOS after *each* example
    # ------------------------------------------------------------------
    flat_ids = []
    eos_id = tok.eos_token_id
    for ids in enc["input_ids"]:
        flat_ids.extend(ids)
        if eos_id is not None:
            flat_ids.append(eos_id)        # one EOS per QA pair

    return flat_ids

# -------------------------------------------------
# 2. Pack into fixed-size blocks (no cross-doc bleed)
# -------------------------------------------------
def pack_ids_to_blocks(ids, block_size):
    """
    ids: list[int]
    Returns: Dataset of dicts with `input_ids` and `attention_mask`
    """
    blocks = []
    # Drop the tail that is shorter than block_size
    usable = len(ids) - (len(ids) % block_size)
    ids = ids[:usable]

    for i in range(0, usable, block_size):
        chunk = ids[i: i + block_size]
        blocks.append({
            "input_ids": chunk,
            "attention_mask": [1] * len(chunk),
            # labels = input_ids for causal LM training
            "labels": chunk.copy(),
        })
    return Dataset.from_list(blocks)

# -------------------------------------------------
# 3. Build train / eval splits
# -------------------------------------------------
# raw_texts → list[str]  (your existing list of Q/A strings)
dataset = Dataset.from_list([{"text": t} for t in raw_texts])

train_val = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_val["train"]
eval_dataset  = train_val["test"]

# ------------------------------------------------------------------
# IMPORTANT: convert the column to a *plain Python list* before tokenising
# ------------------------------------------------------------------
train_ids = tokenize_append_eos(list(train_dataset["text"]))
eval_ids  = tokenize_append_eos(list(eval_dataset["text"]))

# Pack
train_dataset = pack_ids_to_blocks(train_ids, BLOCK_SIZE)
eval_dataset  = pack_ids_to_blocks(eval_ids,  BLOCK_SIZE)

print(
    f"Prepared train_set len {len(train_dataset)} "
    f"and eval_set len {len(eval_dataset)} "
    f"packed training chunks of {BLOCK_SIZE} tokens ✅"
)


Prepared train_set len 1468 and eval_set len 164 packed training chunks of 1024 tokens ✅


# Init Trainer Params

In [9]:
import os
from transformers import TrainingArguments, EarlyStoppingCallback, Trainer
from trl import SFTTrainer

# 1) Absolute, writable, persistent output dir
OUTPUT_DIR = "/storage/models/wtk-qwen3-8b-abliterated-qa-lora-v7"

# 2) Build explicit TrainingArguments (NO dict here)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    overwrite_output_dir=True,
    resume_from_checkpoint=True,  # Fresh start unless resuming
    num_train_epochs=2,  # Increased: 600 MB (~300K–600K tokens) needs 1–2 passes
    per_device_train_batch_size=16,  
    gradient_accumulation_steps=2,  # Balances batch size (~8 effective)
    lr_scheduler_type="cosine",  # Keep: stable for esoteric data
    learning_rate=5e-6,  # Slightly higher: suits smaller dataset, lora_alpha=32
    warmup_ratio=0.1,  # Adjusted: gradual ramp for stability
    weight_decay=0.05,  # Keep: prevents overfitting
    fp16=False,  # Keep: A4000 supports BF16
    bf16=True,  # Keep: efficient on Ampere
    logging_steps=10,  # Keep: frequent monitoring
    eval_strategy="steps",
    eval_steps=10,                     # every ~400 examples
    save_steps=30,                     # Keep: regular checkpoints
    save_total_limit=2,                 # keep best + final only
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    remove_unused_columns=False,
    max_grad_norm=0.3,  # Lowered: stabilizes expanded modules
    dataloader_num_workers=4,  # Keep: stable on mounted storage
)

# Add EarlyStopping: stop if no improvement for 3 evaluations
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=3,   # Wait 3 eval steps (150 steps total)
    early_stopping_threshold=0.001  # Optional: min improvement
)

# Create eval dataset 


# 3) Build the trainer
#trainer = SFTTrainer(
#    model=peft_model,
#    tokenizer=tok,
#    train_dataset=train_dataset,
#    max_seq_length=BLOCK_SIZE,
#    args=training_args,
#)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tok,
    callbacks=[early_stopping],  # ← ADD HERE
)

print(f"Created SFTTrainer ✅")


Created SFTTrainer ✅


/tmp/ipykernel_587/1085983408.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


# Train using SFTTrainer (new)

In [10]:
# 4) Start training (this might take a while)
trainer.train()
print("Training complete ✅")

# 5) Save trained model to storage
trainer.model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,468 | Num Epochs = 2 | Total steps = 92
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 2 x 1) = 32
 "-____-"     Trainable parameters = 15,335,424 of 8,206,070,784 (0.19% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
10,1.890000,1.893509
20,1.873000,1.892012
30,1.883300,1.889323
40,1.875200,1.885486
50,1.877000,1.881735
60,1.865800,1.878742
70,1.876700,1.876858
80,1.858600,1.876002
90,1.862900,1.875910


Unsloth: Not an error, but Qwen3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Training complete ✅
Training results saved ✅


# Resume training using SFTTrainer (only use if resuming)

In [6]:
CHECKPOINT_DIR = OUTPUT_DIR + "/" + "checkpoint-500"

#4) Start training (this might take a while)
print(f"Resuming from checkpoint: {CHECKPOINT_DIR}")
trainer.train(resume_from_checkpoint=CHECKPOINT_DIR)
print("Training complete ✅")

# 5) Save trained model to storage
trainer.model.save_pretrained(OUTPUT_DIR)
tok.save_pretrained(OUTPUT_DIR)

print("Training results saved ✅")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Resuming from checkpoint: /workspace/wtk-qwen3-8b-health-lora-v4/checkpoint-500


ValueError: Can't find a valid checkpoint at /workspace/wtk-qwen3-8b-health-lora-v4/checkpoint-500

# Push LoRa to Huggingface

In [11]:
from huggingface_hub import HfApi, upload_folder

repo_id = "peers-ai/wtk-qwen3-8b-abliterated-qa-lora-v7"
folder = "/storage/models/wtk-qwen3-8b-abliterated-qa-lora-v7"  # contains adapter_config.json & adapter_model.bin

api = HfApi()
# create the repo if it doesn't exist
api.create_repo(repo_id, repo_type="model", private=True, exist_ok=True)

# upload all files in the folder
upload_folder(
    repo_id=repo_id,
    folder_path=folder,
    repo_type="model",
)
print(f"✅ Uploaded to https://huggingface.co/{repo_id}")


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✅ Uploaded to https://huggingface.co/peers-ai/wtk-qwen3-8b-abliterated-qa-lora-v7
